In [68]:
#!python -m pip install --upgrade pip
#%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
#%pip install "aif360[Reductions, inFairness]"

In [69]:
import json
import pandas as pd
from pprint import pprint
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, BatchNormalization # type: ignore

from aif360.datasets import BinaryLabelDataset, StandardDataset
from aif360.algorithms.preprocessing import LFR
from fairlearn.preprocessing import CorrelationRemover
from aif360.algorithms.inprocessing import GerryFairClassifier, PrejudiceRemover, MetaFairClassifier
from aif360.algorithms.postprocessing import EqOddsPostprocessing, RejectOptionClassification

random_seed = 15

In [70]:
PATH = 'C:/Users/andre/Desktop/ProjectWork_AEQUITAS_AKKODIS/'
df = (
    pd.read_excel(PATH + 'data/Dataset_Preprocessed.xlsx')
)
df.head()

,Technical Skills,Comunication,Maturity,Dynamism,Mobility,English,Candidate State_encoded,Event_Feedback_encoded,Residence City_encoded,Residence Province_encoded,...,Sector_encoded,Job Family Hiring_encoded,Job Title Hiring_encoded,Overall_encoded,Minimum Ral_encoded,Ral Maximum_encoded,Study Level_encoded,Current Ral_encoded,Expected Ral_encoded,Status_encoded
0,3,2,3,3,2,4,6,11,689,102,...,1,3,5,5,7,9,6,4,5,1
1,2,2,2,2,2,3,4,10,218,8,...,0,4,8,0,1,1,0,5,6,0
2,2,2,2,2,2,3,3,9,157,22,...,2,4,8,0,1,1,0,1,1,1
3,2,2,2,2,2,3,3,11,636,57,...,13,4,8,0,1,1,0,3,3,1
4,3,2,2,3,1,1,2,11,39,98,...,1,4,8,3,1,1,0,1,5,0


In [71]:
with open(PATH + 'data/encoding_mappings.json', 'r') as f:
    encoding_mappings = json.load(f)
pprint(encoding_mappings)

{'Age Range': {'20 - 25 years': 1,
               '26 - 30 years': 2,
               '31 - 35 years': 3,
               '36 - 40 years': 4,
               '40 - 45 years': 5,
               '< 20 years': 0,
               '> 45 years': 6},
 'Candidate State': {'Economic proposal': 5,
                     'First contact': 1,
                     'Hired': 6,
                     'Imported': 0,
                     'In selection': 2,
                     'QM': 3,
                     'Vivier': 4},
 'Current Ral': {'+ 50 K': 18,
                 '- 20 K': 2,
                 '20-22 K': 3,
                 '22-24 K': 4,
                 '24-26 K': 5,
                 '26-28 K': 6,
                 '28-30 K': 7,
                 '30-32 K': 8,
                 '32-34 K': 9,
                 '34-36 K': 10,
                 '36-38 K': 11,
                 '38-40 K': 12,
                 '40-42 K': 13,
                 '42-44 K': 14,
                 '44-46 K': 15,
                 '46-48 K': 16

## Train

### Dataset Preparation

In [72]:
df = shuffle(df, random_state=random_seed)

X = df.copy()
y = df['Status_encoded']
s = df['Sex_encoded']
X_train_split, X_test_split, y_train_split, y_test_split, s_train_split, s_test_split = train_test_split(X, y, s, test_size=0.2, random_state=random_seed, stratify=y)

In [80]:
train_df = X_train_split.copy()
train_df['target'] = y_train_split.values
train_df['sex'] = s_train_split.values

train_ds = StandardDataset(
    train_df,
    label_name='target', # the column with labels (0/1)
    favorable_classes=[1], # value considered positive
    protected_attribute_names=['sex'], # or ['race'], etc.
    privileged_classes=[[1]] # e.g., male if 1 = male
)
train_df.head()

,Technical Skills,Comunication,Maturity,Dynamism,Mobility,English,Candidate State_encoded,Event_Feedback_encoded,Residence City_encoded,Residence Province_encoded,...,Job Title Hiring_encoded,Overall_encoded,Minimum Ral_encoded,Ral Maximum_encoded,Study Level_encoded,Current Ral_encoded,Expected Ral_encoded,Status_encoded,target,sex
1459,2,2,2,2,2,3,3,11,721,57,...,8,0,1,1,0,17,14,1,1,1
1947,2,2,2,3,3,3,2,11,689,102,...,8,3,1,1,0,6,8,0,0,0
1506,2,2,2,2,2,3,3,10,707,108,...,8,0,1,1,0,1,1,1,1,1
1905,3,2,3,2,1,3,6,11,147,17,...,5,3,1,1,6,1,1,1,1,1
1957,1,2,2,2,1,3,2,11,54,53,...,8,3,1,1,0,1,1,0,0,1


In [81]:
test_df = X_test_split.copy()
test_df['target'] = y_test_split.values
test_df['sex'] = s_test_split.values

test_ds = StandardDataset(
    test_df,
    label_name='target', # the column with labels (0/1)
    favorable_classes=[1], # value considered positive
    protected_attribute_names=['sex'], # or ['race'], etc.
    privileged_classes=[[1]] # e.g., male if 1 = male
)
test_df.head()

,Technical Skills,Comunication,Maturity,Dynamism,Mobility,English,Candidate State_encoded,Event_Feedback_encoded,Residence City_encoded,Residence Province_encoded,...,Job Title Hiring_encoded,Overall_encoded,Minimum Ral_encoded,Ral Maximum_encoded,Study Level_encoded,Current Ral_encoded,Expected Ral_encoded,Status_encoded,target,sex
2545,2,2,2,2,2,3,2,10,565,8,...,8,0,1,1,0,0,0,0,0,1
2393,1,3,2,2,2,3,2,6,394,57,...,8,1,1,1,0,0,0,0,0,1
1514,3,3,3,2,1,3,2,11,435,60,...,8,5,1,1,0,1,1,0,0,1
1003,2,2,2,2,2,3,2,10,534,8,...,8,0,1,1,0,1,0,0,0,1
2493,3,2,3,2,3,3,2,6,116,17,...,8,3,1,1,0,1,1,0,0,1


In [82]:
predictions = {}
print(f"Train dataset: {train_ds.feature_names}")
print(f"Train dataset: {train_ds.protected_attribute_names}")
print(f"Train dataset: {train_ds.label_names}")
print(f"Test dataset: {test_ds.feature_names}")
print(f"Test dataset: {test_ds.protected_attribute_names}")
print(f"Test dataset: {test_ds.label_names}")


Train dataset: ['Technical Skills', 'Comunication', 'Maturity', 'Dynamism', 'Mobility', 'English', 'Candidate State_encoded', 'Event_Feedback_encoded', 'Residence City_encoded', 'Residence Province_encoded', 'Residence Region_encoded', 'Residence State_encoded', 'European Residence_encoded', 'Italian Residence_encoded', 'Age Range_encoded', 'Sex_encoded', 'Protected Category_encoded', 'Study Area_encoded', 'Study Title_encoded', 'Years Experience_encoded', 'Sector_encoded', 'Job Family Hiring_encoded', 'Job Title Hiring_encoded', 'Overall_encoded', 'Minimum Ral_encoded', 'Ral Maximum_encoded', 'Study Level_encoded', 'Current Ral_encoded', 'Expected Ral_encoded', 'Status_encoded', 'sex']
Train dataset: ['sex']
Train dataset: ['target']
Test dataset: ['Technical Skills', 'Comunication', 'Maturity', 'Dynamism', 'Mobility', 'English', 'Candidate State_encoded', 'Event_Feedback_encoded', 'Residence City_encoded', 'Residence Province_encoded', 'Residence Region_encoded', 'Residence State_enc

### Pre-Processing

In [97]:
lfr = LFR(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}],
    k=5, Ax=0.01, Ay=1.0, Az=50.0, verbose=0,
    print_interval=250, seed=None
)

train_lfr_ds = lfr.fit_transform(train_ds)
X_train_lfr_df = pd.DataFrame(train_lfr_ds.features)

test_lfr_ds = lfr.transform(test_ds)
X_test_lfr_df = pd.DataFrame(test_lfr_ds.features)

In [91]:
clf_lfr = LogisticRegression(solver='liblinear')
clf_lfr.fit(X_train_lfr_df, y_train_split)
preds_lfr = clf_lfr.predict(X_test_lfr_df)

predictions['lfr'] = preds_lfr

In [87]:
cr = CorrelationRemover(sensitive_feature_ids=['sex'], alpha=1)

X_train_cr = cr.fit_transform(train_df)
X_train_cr_df = pd.DataFrame(X_train_cr)

X_test_cr = cr.transform(test_df)
X_test_cr_df = pd.DataFrame(X_test_cr)

In [ ]:
clf_cr = LogisticRegression(solver='liblinear')
clf_cr.fit(X_train_cr_df, y_train_split)
preds_lfr = clf_cr.predict(X_test_cr_df)

predictions['cr'] = preds_lfr

In [ ]:
print(predictions)

### In-Processing

In [98]:
gfc = GerryFairClassifier(
    C=10,
    gamma=0.01,
    fairness_def='FP',
    max_iters=10,
    printflag=False,
    heatmapflag=False,
    heatmap_iter=10,
    heatmap_path='.',
    predictor=LinearRegression()
)
gfc.fit(train_ds)
pred_gfc = gfc.predict(test_ds)

predictions['gfc'] = pred_gfc.labels.ravel()

In [99]:
pr = PrejudiceRemover(sensitive_attr='sex', class_attr='target', eta=1.0)
pr = pr.fit(train_ds_bin)
pred_pr = pr.predict(test_ds)

predictions['pr'] = pred_pr.labels

c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\aif360\algorithms\inprocessing\prejudice_remover.py:208: UserWarning: loadtxt: input contained no data: "C:\Users\andre\AppData\Local\Temp\tmp4rvkcp_c"
  m = np.loadtxt(output_name)


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [101]:
mfc = MetaFairClassifier(
    sensitive_attr='sex',
    tau=0.8,
    type='fdr',
    seed=None
)
mfc.fit(train_ds_bin)
pred_mfc = mfc.predict(test_ds_bin)

predictions['mfc'] = pred_mfc.labels

c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\aif360\algorithms\inprocessing\celisMeta\FalseDiscovery.py:31: RuntimeWarning: invalid value encountered in divide
  prob_y_1 = (prob_1_1 + prob_1_0) / total
c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\aif360\algorithms\inprocessing\celisMeta\FalseDiscovery.py:32: RuntimeWarning: invalid value encountered in divide
  prob_z_0 = (prob_m1_0 + prob_1_0) / total
c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\aif360\algorithms\inprocessing\celisMeta\FalseDiscovery.py:33: RuntimeWarning: invalid value encountered in divide
  prob_z_1 = (prob_m1_1 + prob_1_1) / total
c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\aif360\algorithms\inprocessing\celisMeta\FalseDiscovery.py:35: RuntimeWarning: invalid value encountered in divide
  probc_m1_0 = prob_m1_0 / total
c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packa

TypeError: cannot unpack non-iterable NoneType object

In [102]:
print(predictions)

{'lfr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

### Post-Processing

In [103]:
clf = LogisticRegression(solver='liblinear')
clf.fit(X_train_cr_df, y_train_split)
preds_lfr = clf.predict(X_test_cr_df)

In [104]:
eop = EqOddsPostprocessing(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)
eop = eop.fit(train_ds, preds_lfr)
pred_eop = eop.predict(test_ds)

predictions['eop'] = pred_eop.labels

TypeError: 'classified_dataset' should be a BinaryLabelDataset or a MulticlassLabelDataset.

In [105]:
clf = LogisticRegression(solver='liblinear')
clf.fit(X_train_cr_df, y_train_split)
preds_lfr = clf.predict(X_test_cr_df)

In [106]:
roc = RejectOptionClassification(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}],
    low_class_thresh=0.01,
    high_class_thresh=0.99,
    num_class_thresh=100,
    metric_name='Average odds difference'
)
roc = roc.fit(train_ds, preds_lfr)
pred_roc = roc.predict(test_ds)

predictions['roc'] = pred_roc.labels
print(predictions["roc"])

TypeError: copy() got an unexpected keyword argument 'deepcopy'

### Models

In [107]:
def create_model(seed):
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Dense(128, input_dim=22, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'KNN': KNeighborsClassifier(),
    'Neural Network': create_model(random_seed),
    
    'LFR': clf_lfr,
    'Correlation Remover' : clf_cr,
    'GerryFair Classifier' : gfc,
    'Prejudice Remover' : pr,
    'MetaFair Classifier' : mfc,
    'Eq Odds Postprocessing' : eop,
    'Reject Option Classification' : roc,    
}

In [ ]:
for name, model in models.items():
    if name in ['Linear Regression', 'Decision Tree', 'Naive Bayes', 'XGBoost', 'KNN']:
        model.fit(X_train_split, y_train_split)
        print(f"Model {name} trained!") 
    elif name in ['Neural Network']:
        model.fit(X_train_split, y_train_split, epochs=15, batch_size=64, validation_split=0.2)
        print(f"Model {name} trained!") 
    else:
        print(f"Model {name} already trained.") 
        continue

    y_pred = model.predict(X_test_split)

    if name in ['Linear Regression', 'XGBoost', 'Neural Network']:
        y_pred = (y_pred > 0.5).astype(int)

    predictions[name] = y_pred

In [ ]:
with open(PATH + 'data/models.json', 'w') as f:
    json.dump(models, f, indent=4)

## Dataframe

In [100]:
predictions_df = pd.DataFrame({
    'Linear Regression' : predictions['Linear Regression'],
    'Decision Tree' : predictions['Decision Tree'],
    'Naive Bayes' : predictions['Naive Bayes'],
    'XGBoost' : predictions['XGBoost'],
    'kNN' : predictions['KNN'],
    'Neural Network' : predictions['Neural Network'],

    'LFR' : predictions['lfr'],
    'Correlation Remover' : predictions['cr'],
    'GerryFair Classifier' : predictions['gfc'],
    'Prejudice Remover' : predictions['pr'],
    'MetaFair Classifier' : predictions['mfc'],
    'Eq Odds Postprocessing' : predictions['eop'],
    'Reject Option Classification' : predictions['roc'],

})

y_test = {
    'Reference Full' : y_test_full,
    'Reference' : y_test,
}

KeyError: 'Linear Regression'

In [ ]:
with open(PATH + 'data/predictions_df.json', 'w') as f:
    json.dump(predictions_df, f, indent=4)
with open(PATH + 'data/y_test.json', 'w') as f:
    json.dump(y_test, f, indent=4)